In [ ]:
import tiktoken
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document
from datasets import load_dataset
from openai import OpenAI

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

import gradio as gr



In [72]:
load_dotenv()

MODEL = "gpt-4.1-nano"
EMBED_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "squad_rag"

api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
dataset = load_dataset("squad", split="train[:500]")
print(dataset)
print(dataset[0].keys())

In [ ]:
docs = []
for i, row in enumerate(dataset):
    docs.append(
        {
            "id": f"squad_{i}",
            "text": row["context"],
            "metadata": {
                "title": row["title"],
                "question": row["question"],
                "source_id": f"squad_{i}",
            },
        }
    )

print("docs:", len(docs))
print(docs[0])

In [65]:
enc = tiktoken.get_encoding("cl100k_base")

def chunk_text_token_based(text: str, chunk_size: int, overlap: int):
    tokens = enc.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(enc.decode(chunk_tokens))
        if end >= len(tokens):
            break
        start = end - overlap
    return chunks

In [ ]:
chunks = []
for doc in docs:
    parts = chunk_text_token_based(doc["text"], chunk_size=200, overlap=40)
    for i, part in enumerate(parts):
        chunks.append(
            {
                "id": f'{doc["id"]}_chunk_{i}',
                "text": part,
                "metadata": {**doc["metadata"], "chunk_index": i},
            }
        )

print("chunks:", len(chunks))
print(chunks[0])

In [ ]:
lc_docs = [
    Document(
        page_content=c["text"],
        metadata={**c["metadata"], "chunk_id": c["id"]},
    )
    for c in chunks
]

print("lc_docs:", len(lc_docs))

In [ ]:
if api_key:
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL, api_key=api_key)
    print("Using OpenAI embeddings")
else:
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    print("Using HuggingFace embeddings fallback")

In [ ]:
vectorstore = Chroma.from_documents(
    documents=lc_docs,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store ready")

In [ ]:
# build vectors from current chunks
texts = [c["text"] for c in chunks]
vectors = embeddings.embed_documents(texts)

print("num vectors:", len(vectors))
print("vector dim:", len(vectors[0]) if vectors else 0)

In [ ]:


# vectors: list of embedding vectors, same order as chunks
X = np.array(vectors)

# reduce to 3D
pca = PCA(n_components=3, random_state=42)
X3 = pca.fit_transform(X)

# build dataframe for plotting
df = pd.DataFrame({
    "x": X3[:, 0],
    "y": X3[:, 1],
    "z": X3[:, 2],
    "title": [c["metadata"].get("title", "N/A") for c in chunks],
    "chunk_id": [c["id"] for c in chunks],
    "text_preview": [c["text"][:120].replace("\n", " ") for c in chunks],
})

fig = px.scatter_3d(
    df,
    x="x",
    y="y",
    z="z",
    color="title",  # or remove if too many unique titles
    hover_data=["chunk_id", "text_preview"],
    title="3D Embedding Projection (PCA)"
)
fig.update_traces(marker=dict(size=3, opacity=0.75))
fig.show()

In [ ]:
query = "What happened to Beyonce in the Super Bowl halftime show?"
retrieved = retriever.invoke(query)

print(f"Retrieved {len(retrieved)} chunks\n")
for i, d in enumerate(retrieved, 1):
    print(f"[{i}] title={d.metadata.get('title')} chunk={d.metadata.get('chunk_index')}")
    print(d.page_content[:220], "...\n")

In [73]:
def answer_with_rag(question: str):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n---\n\n".join([d.page_content for d in retrieved_docs])

    if not api_key:
        return {
            "answer": "No OPENAI_API_KEY found. Retrieval works, but generation is disabled.",
            "contexts": retrieved_docs,
        }

    llm = ChatOpenAI(model=MODEL, api_key=api_key, temperature=0)

    prompt = f"""
You are a QA assistant.
Answer using the context.
If the context is clearly relevant, provide the best possible answer in 1-3 sentences.
Only say "I don't know" when the context is empty or unrelated.

Question:
{question}

Context:
{context}
"""

    response = llm.invoke(prompt)
    return {"answer": response.content, "contexts": retrieved_docs}

In [ ]:

def rag_chat(question):
    question = (question or "").strip()
    if not question:
        return "Please enter a question.", ""

    result = answer_with_rag(question)
    answer = result.get("answer", "")

    # show top retrieved chunks for debugging/testing
    contexts = result.get("contexts", [])
    context_preview = []
    for i, d in enumerate(contexts, 1):
        title = d.metadata.get("title", "N/A")
        chunk_idx = d.metadata.get("chunk_index", "N/A")
        snippet = d.page_content[:300].replace("\n", " ")
        context_preview.append(f"[{i}] title={title}, chunk={chunk_idx}\n{snippet}...")

    return answer, "\n\n".join(context_preview)

demo = gr.Interface(
    fn=rag_chat,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Ask a question about the indexed SQuAD contexts...",
        label="Question",
    ),
    outputs=[
        gr.Textbox(label="RAG Answer"),
        gr.Textbox(label="Retrieved Contexts (Debug)"),
    ],
    title="SQuAD RAG Tester",
    description="Type a question, run retrieval + generation, inspect retrieved chunks.",
    allow_flagging="never",
)

demo.launch()